# 프로젝트 4: 에너지관리 - 에너지 원단위 & 효율 분석 

## 프로젝트 배경

당신은 **한국정밀산업(주)** 에너지관리팀의 데이터 분석가입니다.  
최근 전기요금이 대폭 인상되면서 경영진이 **에너지 절감 방안**을 긴급히 요구하고 있습니다.

> *"올해 전기요금이 작년 대비 25% 올랐습니다.  
> A라인의 전력 소비가 유독 높은데, 원인이 뭔가요?  
> C라인은 3월에 인버터를 도입했는데 효과가 있나요?  
> 대기전력 낭비와 피크 시간대 전력 집중 문제도 분석해 주세요."*

---

- 한 줄 한 줄 **주석**으로 설명합니다
- 복잡한 코드는 여러 단계로 **나눠서** 작성합니다
- 처음 배우는 분도 **그대로 따라하면** 결과가 나옵니다

## Part 0: 환경 설정 및 데이터 로드

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd       # 데이터 분석용
import numpy as np        # 수학 계산용
import matplotlib.pyplot as plt  # 그래프 그리기용
import seaborn as sns     # 예쁜 그래프용
from scipy import stats   # 통계 검정용
import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 숨기기

# 한글 폰트 설정 (그래프에 한글 표시)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
sns.set_style('whitegrid')  # 그래프 배경 스타일

In [ ]:
# CSV 파일 4개 불러오기
equip = pd.read_csv('../data/project4/p4_equipment.csv', encoding='utf-8-sig')    # 설비 정보
energy = pd.read_csv('../data/project4/p4_energy_log.csv', encoding='utf-8-sig')  # 에너지 사용 로그
prod = pd.read_csv('../data/project4/p4_production_log.csv', encoding='utf-8-sig') # 생산 로그
tariff = pd.read_csv('../data/project4/p4_tariff.csv', encoding='utf-8-sig')      # 전기 요금표

# 날짜 컬럼을 날짜 형식으로 변환
energy['timestamp'] = pd.to_datetime(energy['timestamp'])
prod['date'] = pd.to_datetime(prod['date'])

print('데이터 로드 완료!')
print(f'설비: {len(equip)}건')
print(f'에너지: {len(energy):,}건')
print(f'생산: {len(prod):,}건')
print(f'요금: {len(tariff)}건')

---
## Part 1: 데이터 탐색 및 전처리 (15점)

### 문제 1-1: 데이터 탐색 (5점)

4개 데이터의 구조를 확인합니다.

In [ ]:
# 설비 데이터 확인
print('=== 설비 데이터 ===')
print(f'크기: {equip.shape}')  # (행 수, 열 수)
print()
equip.head()  # 상위 5개 행 보기

In [ ]:
# 에너지 데이터 확인
print('=== 에너지 데이터 ===')
print(f'크기: {energy.shape}')
print(f'기간: {energy["timestamp"].min()} ~ {energy["timestamp"].max()}')
print()
energy.head()

In [ ]:
# 생산 데이터 확인
print('=== 생산 데이터 ===')
print(f'크기: {prod.shape}')
print()
prod.head()

In [ ]:
# 요금 단가표 확인
print('=== 전기 요금 단가표 (TOU: Time of Use) ===')
print('→ 시간대별로 전기 요금이 다릅니다')
print()
tariff

In [ ]:
# 결측치(빈 값) 확인
print('=== 에너지 데이터 결측치 ===')
print(energy.isnull().sum())  # 각 컬럼별 결측치 개수

print()
print('=== 생산 데이터 결측치 ===')
print(prod.isnull().sum())

In [ ]:
# 설비 에너지 프로파일 (인버터 유무, 에너지등급 확인)
print('=== 설비별 에너지 프로파일 ===')
print(equip[['equipment_id', 'line', 'rated_power_kw', 'has_inverter', 
             'standby_power_kw', 'energy_grade']].to_string(index=False))
print()
print('→ A라인: 인버터 없음, 에너지등급 4~5등급 (비효율)')
print('→ B라인: 인버터 있음, 에너지등급 1~2등급 (효율적)')

### 문제 1-2: 결측치 처리 (5점)

In [ ]:
# 결측치 처리 전 개수 확인
print('처리 전 결측치:')
print(f'  energy - power_kwh: {energy["power_kwh"].isnull().sum()}개')
print(f'  energy - ambient_temp_c: {energy["ambient_temp_c"].isnull().sum()}개')
print(f'  prod - production_qty: {prod["production_qty"].isnull().sum()}개')
print(f'  prod - operating_hours: {prod["operating_hours"].isnull().sum()}개')

In [ ]:
# [1] energy 데이터: 설비별로 시간순 정렬 후 보간(interpolate)
#     → 앞뒤 값의 중간값으로 빈 칸을 채우는 방법

energy = energy.sort_values(['equipment_id', 'timestamp']).reset_index(drop=True)

# power_kwh: 설비별로 보간
energy['power_kwh'] = energy.groupby('equipment_id')['power_kwh'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

# ambient_temp_c: 전체 시간순으로 보간 (온도는 공장 전체 공통)
energy['ambient_temp_c'] = energy['ambient_temp_c'].interpolate(method='linear').ffill().bfill()

print('energy 결측치 처리 완료!')

In [ ]:
# [2] prod 데이터: 설비별 평균값으로 빈 칸 채우기

prod['production_qty'] = prod.groupby('equipment_id')['production_qty'].transform(
    lambda x: x.fillna(x.mean())
)

prod['operating_hours'] = prod.groupby('equipment_id')['operating_hours'].transform(
    lambda x: x.fillna(x.mean())
)

print('prod 결측치 처리 완료!')

In [ ]:
# 결측치 처리 후 확인
print('처리 후 결측치:')
print(f'  energy: {energy.isnull().sum().sum()}개')
print(f'  prod: {prod[["production_qty","operating_hours"]].isnull().sum().sum()}개')
print()
print('→ 모든 결측치가 처리되었습니다!')

### 문제 1-3: 파생 컬럼 추가 (5점)

분석에 필요한 새로운 컬럼을 만듭니다.

In [ ]:
# [1단계] 날짜/시간 관련 컬럼 추가
energy['date'] = energy['timestamp'].dt.date         # 날짜만
energy['hour'] = energy['timestamp'].dt.hour         # 시간 (0~23)
energy['month'] = energy['timestamp'].dt.month       # 월 (1~6)
energy['weekday'] = energy['timestamp'].dt.weekday   # 요일 (0=월, 6=일)

print('날짜/시간 컬럼 추가 완료')
energy[['timestamp', 'date', 'hour', 'month', 'weekday']].head(3)

In [ ]:
# [2단계] 라인 컬럼 추가
# equipment_id에서 라인 정보 추출 (EQ-A01 → A → A라인)

def get_line(equipment_id):
    """설비 ID에서 라인명 추출하는 함수"""
    letter = equipment_id[3]   # 'EQ-A01'에서 'A' 추출
    if letter == 'A':
        return 'A라인'
    elif letter == 'B':
        return 'B라인'
    else:
        return 'C라인'

energy['line'] = energy['equipment_id'].apply(get_line)

print('라인 컬럼 추가 완료')
print(energy['line'].value_counts())

In [ ]:
# [3단계] 시간대(TOU) 분류 컬럼 추가
# 전기 요금은 시간대에 따라 다릅니다:
#   경부하(저렴): 23시~09시
#   중간부하(보통): 09시, 12시, 17~22시
#   최대부하(비쌈): 10~11시, 13~16시

def get_time_zone(hour):
    """시간을 보고 요금 시간대를 분류하는 함수"""
    if hour >= 23 or hour < 9:
        return '경부하'        # 야간: 가장 저렴
    elif hour in [10, 11, 13, 14, 15, 16]:
        return '최대부하'      # 피크: 가장 비쌈
    else:
        return '중간부하'      # 나머지: 중간

energy['time_zone'] = energy['hour'].apply(get_time_zone)

print('시간대 분류 완료')
print(energy['time_zone'].value_counts())

In [ ]:
# [4단계] 전기 요금 단가 컬럼 추가
# 계절(하계/춘추계) + 시간대에 따라 단가가 달라짐

def get_tariff_rate(row):
    """월과 시간대로 전기 요금 단가를 구하는 함수"""
    month = row['month']
    time_zone = row['time_zone']
    
    if month in [6, 7, 8]:  # 하계 (여름)
        if time_zone == '경부하':
            return 65
        elif time_zone == '중간부하':
            return 105
        else:  # 최대부하
            return 150
    else:  # 춘추계 (봄/가을/겨울)
        if time_zone == '경부하':
            return 60
        elif time_zone == '중간부하':
            return 85
        else:  # 최대부하
            return 120

energy['tariff_rate'] = energy.apply(get_tariff_rate, axis=1)

print('전기 요금 단가 추가 완료')
print(energy[['time_zone', 'tariff_rate']].drop_duplicates().sort_values('tariff_rate'))

In [ ]:
# [5단계] 전기요금 계산
# 전기요금 = 전력사용량(kWh) × 단가(원/kWh)

energy['cost_won'] = energy['power_kwh'] * energy['tariff_rate']

print('전기요금 계산 완료!')
print()
# 최종 확인
energy[['timestamp', 'equipment_id', 'line', 'hour', 'time_zone', 
         'tariff_rate', 'power_kwh', 'cost_won']].head(5)

---
## Part 2: 시간대별 전력 사용 패턴 (25점)

### 문제 2-1: 라인별·시간대별 전력 사용 히트맵 (7점)

In [ ]:
# 라인별, 시간별 평균 전력 구하기
# → pivot_table 대신 groupby + unstack 사용
heatmap_data = energy.groupby(['line', 'hour'])['power_kwh'].mean().unstack()

# 히트맵 그리기
plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_data,         # 데이터
            annot=True,            # 숫자 표시
            fmt='.1f',             # 소수점 1자리
            cmap='YlOrRd',         # 색상: 노랑→빨강
            linewidths=0.5)        # 칸 구분선

plt.title('라인별·시간대별 평균 전력 사용량 (kWh)', fontsize=14)
plt.xlabel('시간 (0~23시)')
plt.ylabel('')
plt.tight_layout()
plt.show()

print('→ A라인이 전체적으로 가장 진한 색 = 전력 소비가 가장 높음')
print('→ 모든 라인에서 10~16시(최대부하)에 전력이 집중됨')

In [ ]:
# 라인별 전력이 가장 높은 시간 / 낮은 시간 확인
for line in ['A라인', 'B라인', 'C라인']:
    row = heatmap_data.loc[line]
    print(f'{line}: 최대 {row.idxmax()}시 ({row.max():.1f}kWh) / 최소 {row.idxmin()}시 ({row.min():.1f}kWh)')

### 문제 2-2: 가동 vs 비가동 전력 비교 - 대기전력 분석 (6점)

In [ ]:
# 설비별로 가동(True) / 비가동(False) 시 평균 전력 구하기
standby_analysis = energy.groupby(['equipment_id', 'is_operating'])['power_kwh'].mean().unstack()
standby_analysis.columns = ['비가동(대기전력)', '가동전력']

# 대기전력 비율 = 비가동 전력 / 가동 전력 × 100
standby_analysis['대기전력비율(%)'] = (
    standby_analysis['비가동(대기전력)'] / standby_analysis['가동전력'] * 100
).round(1)

# 설비 정보(라인, 인버터 유무) 합치기
standby_analysis = standby_analysis.merge(
    equip[['equipment_id', 'line', 'has_inverter', 'energy_grade']],
    left_index=True, right_on='equipment_id'
)

# 대기전력이 높은 순서로 정렬
standby_analysis = standby_analysis.sort_values('비가동(대기전력)', ascending=False)

print('=== 설비별 대기전력 분석 ===')
standby_analysis

In [ ]:
# 대기전력 바 차트
plt.figure(figsize=(12, 6))

# 대기전력이 낮은 순서로 정렬 (위에서 아래로)
sorted_data = standby_analysis.sort_values('비가동(대기전력)', ascending=True)

# 라인별 색상 지정
line_colors = {'A라인': '#e74c3c', 'B라인': '#2ecc71', 'C라인': '#3498db'}
colors = [line_colors[l] for l in sorted_data['line']]

# 수평 바 차트 그리기
plt.barh(sorted_data['equipment_id'], sorted_data['비가동(대기전력)'], color=colors)

plt.xlabel('비가동 시 평균 전력 (kWh)')
plt.title('설비별 대기전력 비교', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('→ A라인 설비들이 대기전력이 가장 높음')
print('→ A라인: 인버터 없음, 에너지등급 낮음 → 절전모드 도입 우선 대상')

### 문제 2-3: 월별 전력 추이 - C라인 인버터 도입 효과 (6점)

In [ ]:
# 라인별 월별 평균 전력
monthly_power = energy.groupby(['line', 'month'])['power_kwh'].mean().unstack(0)

# 그래프 그리기
plt.figure(figsize=(12, 6))

for line, color in line_colors.items():
    plt.plot(monthly_power.index, monthly_power[line], 'o-', 
             color=color, linewidth=2, markersize=8, label=line)

# C라인 인버터 도입 시점 표시 (3월)
plt.axvline(2.5, color='gray', linestyle='--', alpha=0.5, label='C라인 인버터 도입')

plt.xlabel('월')
plt.ylabel('평균 전력 (kWh/h)')
plt.title('라인별 월별 평균 전력 추이', fontsize=13, fontweight='bold')
plt.xticks(range(1, 7), ['1월', '2월', '3월', '4월', '5월', '6월'])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# C라인 인버터 도입 전(1~2월) vs 후(3~6월) 비교
c_energy = energy[energy['line'] == 'C라인']

# 도입 전: 1~2월
c_before = c_energy[c_energy['month'].isin([1, 2])]['power_kwh']
# 도입 후: 3~6월
c_after = c_energy[c_energy['month'].isin([3, 4, 5, 6])]['power_kwh']

print(f'C라인 인버터 도입 전(1~2월): 평균 {c_before.mean():.2f} kWh')
print(f'C라인 인버터 도입 후(3~6월): 평균 {c_after.mean():.2f} kWh')

# 절감률 계산
saving_rate = (c_before.mean() - c_after.mean()) / c_before.mean() * 100
print(f'절감률: {saving_rate:.1f}%')

# 통계적으로 유의미한 차이인지 t-검정
t_stat, p_value = stats.ttest_ind(c_before, c_after)
print(f'\nt-검정 결과: p-value = {p_value:.6f}')
if p_value < 0.05:
    print('→ p < 0.05이므로 통계적으로 유의미한 차이가 있습니다!')
else:
    print('→ p >= 0.05이므로 유의미한 차이가 없습니다')

### 문제 2-4: 요일별·시간대별 패턴 (6점)

In [ ]:
# 요일별 평균 전력
weekday_names = ['월', '화', '수', '목', '금', '토', '일']
weekday_power = energy.groupby('weekday')['power_kwh'].mean()

plt.figure(figsize=(10, 5))

# 평일은 파랑, 주말은 빨강
colors = ['#3498db'] * 5 + ['#e74c3c', '#e74c3c']

plt.bar(range(7), weekday_power.values, color=colors)
plt.xticks(range(7), weekday_names)
plt.ylabel('평균 전력 (kWh)')
plt.title('요일별 평균 전력', fontsize=12, fontweight='bold')

# 각 바 위에 숫자 표시
for i, v in enumerate(weekday_power.values):
    plt.text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('→ 주말에도 전력이 상당히 사용됨 = 대기전력 낭비')

In [ ]:
# 주말 대기전력 비중 계산
weekend = energy[energy['weekday'].isin([5, 6])]           # 토, 일
weekend_total = weekend['power_kwh'].sum()                 # 주말 총 전력
weekend_standby = weekend[weekend['is_operating'] == False]['power_kwh'].sum()  # 비가동 전력

print(f'주말 총 전력: {weekend_total:,.0f} kWh')
print(f'주말 대기전력: {weekend_standby:,.0f} kWh')
print(f'대기전력 비중: {weekend_standby / weekend_total * 100:.1f}%')

In [ ]:
# 외기온도 vs 전력 관계 확인 (산점도)
# 데이터가 많으니 5000개만 샘플링
sample = energy.dropna(subset=['ambient_temp_c']).sample(5000, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(sample['ambient_temp_c'], sample['power_kwh'], alpha=0.2, s=10)

# 상관계수 계산
corr = sample['ambient_temp_c'].corr(sample['power_kwh'])

plt.xlabel('외기온도 (°C)')
plt.ylabel('전력 사용량 (kWh)')
plt.title(f'외기온도 vs 전력 사용량 (상관계수: {corr:.3f})', fontsize=13)
plt.tight_layout()
plt.show()

print(f'상관계수: {corr:.3f}')
print('→ 외기온도가 올라가면 냉방 부하로 전력 사용이 증가합니다')

---
## Part 3: 에너지 원단위 분석 (25점)

**에너지 원단위** = 제품 1개 만드는 데 사용한 전력 (kWh/개)  
→ 숫자가 낮을수록 에너지 효율이 좋은 것입니다

### 문제 3-1: 에너지 원단위 계산 (7점)

In [ ]:
# [1단계] 설비별·일별 전력 합계 구하기
energy['date_dt'] = pd.to_datetime(energy['date'])
daily_energy = energy.groupby(['equipment_id', 'date_dt'])['power_kwh'].sum().reset_index()
daily_energy.columns = ['equipment_id', 'date', 'daily_power_kwh']

print(f'일별 에너지 데이터: {len(daily_energy)}건')
daily_energy.head()

In [ ]:
# [2단계] 설비별·일별 생산량 합계 구하기
# 컴프레서(x04)는 생산 장비가 아니므로 제외
prod_no_comp = prod[~prod['equipment_id'].str.contains('04')].copy()

daily_prod = prod_no_comp.groupby(['equipment_id', 'date']).agg(
    daily_qty=('production_qty', 'sum'),
    operating_hours=('operating_hours', 'sum')
).reset_index()

print(f'일별 생산 데이터: {len(daily_prod)}건')
daily_prod.head()

In [ ]:
# [3단계] 에너지 + 생산 데이터 합치기
unit_energy = daily_energy.merge(daily_prod, on=['equipment_id', 'date'], how='inner')

# 라인 컬럼 추가
unit_energy['line'] = unit_energy['equipment_id'].apply(get_line)

print(f'결합된 데이터: {len(unit_energy)}건')
unit_energy.head()

In [ ]:
# [4단계] 원단위 계산: 일일전력 / 일일생산량
unit_energy['energy_unit'] = unit_energy['daily_power_kwh'] / unit_energy['daily_qty']

# 이상치 제거 (상위 1%)
q99 = unit_energy['energy_unit'].quantile(0.99)
unit_clean = unit_energy[unit_energy['energy_unit'] <= q99].copy()

print(f'원단위 데이터: {len(unit_clean)}건 (이상치 제거 후)')
print()
print('원단위 기본 통계:')
print(unit_clean['energy_unit'].describe().round(3))

In [ ]:
# 라인별 원단위 분포 히스토그램
plt.figure(figsize=(12, 5))

for line, color in line_colors.items():
    data = unit_clean[unit_clean['line'] == line]['energy_unit']
    mean_val = data.mean()
    plt.hist(data, bins=30, alpha=0.5, color=color, label=f'{line} (평균: {mean_val:.2f})')

plt.xlabel('에너지 원단위 (kWh/개)')
plt.ylabel('빈도')
plt.title('라인별 에너지 원단위 분포', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print('→ A라인: 원단위가 가장 높음 (비효율적)')
print('→ B라인: 원단위가 가장 낮음 (효율적)')

### 문제 3-2: 라인별·설비별 원단위 비교 (6점)

In [ ]:
# 라인별 평균 원단위
line_unit = unit_clean.groupby('line')['energy_unit'].mean().sort_values()

plt.figure(figsize=(8, 5))

colors = [line_colors[l] for l in line_unit.index]
bars = plt.bar(line_unit.index, line_unit.values, color=colors)

# 바 위에 숫자 표시
for bar, val in zip(bars, line_unit.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.ylabel('평균 원단위 (kWh/개)')
plt.title('라인별 평균 에너지 원단위', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 설비별 평균 원단위
equip_unit = unit_clean.groupby('equipment_id')['energy_unit'].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 6))

# 설비별 라인 색상 적용
colors = [line_colors[get_line(eq)] for eq in equip_unit.index]
plt.barh(equip_unit.index, equip_unit.values, color=colors)

# 숫자 표시
for i, (eq, val) in enumerate(equip_unit.items()):
    plt.text(val + 0.02, i, f'{val:.2f}', va='center', fontsize=9)

plt.xlabel('평균 원단위 (kWh/개)')
plt.title('설비별 평균 에너지 원단위', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 문제 3-3: 원단위 월별 추이 - C라인 개선 효과 (6점)

In [ ]:
# 월 컬럼 추가
unit_clean['month'] = unit_clean['date'].dt.month

# 라인별 월별 평균 원단위
monthly_unit = unit_clean.groupby(['line', 'month'])['energy_unit'].mean().unstack(0)

# 그래프 그리기
plt.figure(figsize=(12, 6))

for line, color in line_colors.items():
    plt.plot(monthly_unit.index, monthly_unit[line], 'o-',
             color=color, linewidth=2, markersize=8, label=line)

# C라인 인버터 도입 시점
plt.axvline(2.5, color='gray', linestyle='--', alpha=0.5, label='C라인 인버터 도입')

plt.xlabel('월')
plt.ylabel('평균 원단위 (kWh/개)')
plt.title('라인별 월별 에너지 원단위 추이', fontsize=13, fontweight='bold')
plt.xticks(range(1, 7), ['1월', '2월', '3월', '4월', '5월', '6월'])
plt.legend()
plt.tight_layout()
plt.show()

print('→ C라인: 3월 인버터 도입 후 원단위가 눈에 띄게 감소!')

In [ ]:
# C라인 원단위 개선률 계산
c_unit = unit_clean[unit_clean['line'] == 'C라인']
c_unit_before = c_unit[c_unit['month'].isin([1, 2])]['energy_unit']
c_unit_after = c_unit[c_unit['month'].isin([3, 4, 5, 6])]['energy_unit']

print(f'C라인 원단위 도입 전: {c_unit_before.mean():.3f} kWh/개')
print(f'C라인 원단위 도입 후: {c_unit_after.mean():.3f} kWh/개')
print(f'원단위 개선률: {(c_unit_before.mean() - c_unit_after.mean()) / c_unit_before.mean() * 100:.1f}%')

In [ ]:
# A라인이 B라인 수준으로 개선되면 얼마나 절감할 수 있을까?
a_unit_mean = unit_clean[unit_clean['line'] == 'A라인']['energy_unit'].mean()
b_unit_mean = unit_clean[unit_clean['line'] == 'B라인']['energy_unit'].mean()
a_total_qty = unit_clean[unit_clean['line'] == 'A라인']['daily_qty'].sum()

# 단위당 절감량 × 총 생산량
saving_per_unit = a_unit_mean - b_unit_mean
saving_kwh_annual = saving_per_unit * a_total_qty * 2  # 상반기 → 연간

print(f'=== B라인 벤치마크 절감 시뮬레이션 ===')
print(f'A라인 평균 원단위: {a_unit_mean:.3f} kWh/개')
print(f'B라인 평균 원단위: {b_unit_mean:.3f} kWh/개 (벤치마크)')
print(f'원단위 차이: {saving_per_unit:.3f} kWh/개')
print(f'연간 절감 가능 전력: {saving_kwh_annual:,.0f} kWh')
print(f'연간 절감 금액 (100원/kWh 기준): {saving_kwh_annual * 100 / 10000:,.0f}만원')

### 문제 3-4: 원단위 vs 가동률 관계 (6점)

In [ ]:
# 가동률 = 일일 가동시간 / 24시간 × 100 (%)
unit_clean['utilization'] = unit_clean['operating_hours'] / 24 * 100

# 산점도
plt.figure(figsize=(12, 6))

for line, color in line_colors.items():
    mask = unit_clean['line'] == line
    plt.scatter(unit_clean[mask]['utilization'], unit_clean[mask]['energy_unit'],
                alpha=0.4, s=15, c=color, label=line)

# 상관계수
corr = unit_clean['utilization'].corr(unit_clean['energy_unit'])

plt.xlabel('가동률 (%)')
plt.ylabel('에너지 원단위 (kWh/개)')
plt.title(f'가동률 vs 에너지 원단위 (상관계수: {corr:.3f})', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# 가동률 50% 미만 vs 이상 비교
low_util = unit_clean[unit_clean['utilization'] < 50]
high_util = unit_clean[unit_clean['utilization'] >= 50]
print(f'가동률 < 50%: 평균 원단위 = {low_util["energy_unit"].mean():.3f} kWh/개 ({len(low_util)}건)')
print(f'가동률 >= 50%: 평균 원단위 = {high_util["energy_unit"].mean():.3f} kWh/개 ({len(high_util)}건)')
print('→ 가동률이 낮으면 대기전력 때문에 원단위가 나빠집니다')

---
## Part 4: 전기요금 & 절감 분석 (20점)

### 문제 4-1: 시간대별 요금 구조 분석 (5점)

In [ ]:
# 시간대별 총 전력량과 총 비용 집계
tz_summary = energy.groupby('time_zone').agg(
    총전력량=('power_kwh', 'sum'),
    총비용=('cost_won', 'sum')
).round(0)

# 비중(%) 계산
tz_summary['전력비중(%)'] = (tz_summary['총전력량'] / tz_summary['총전력량'].sum() * 100).round(1)
tz_summary['비용비중(%)'] = (tz_summary['총비용'] / tz_summary['총비용'].sum() * 100).round(1)

# 보기 좋게 정렬
tz_order = ['경부하', '중간부하', '최대부하']
tz_summary = tz_summary.reindex(tz_order)

print('=== 시간대별 전력량 및 비용 ===')
tz_summary

In [ ]:
# 전력 비중 vs 비용 비중 비교 바 차트
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tz_colors = {'경부하': '#2ecc71', '중간부하': '#f39c12', '최대부하': '#e74c3c'}
colors = [tz_colors[tz] for tz in tz_order]

# 왼쪽: 전력 비중
axes[0].bar(tz_order, tz_summary['전력비중(%)'], color=colors)
axes[0].set_ylabel('비중 (%)')
axes[0].set_title('전력량 비중', fontsize=12, fontweight='bold')
for i, val in enumerate(tz_summary['전력비중(%)']):
    axes[0].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# 오른쪽: 비용 비중
axes[1].bar(tz_order, tz_summary['비용비중(%)'], color=colors)
axes[1].set_ylabel('비중 (%)')
axes[1].set_title('비용 비중', fontsize=12, fontweight='bold')
for i, val in enumerate(tz_summary['비용비중(%)']):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('시간대별 전력 비중 vs 비용 비중', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('→ 최대부하: 전력 비중보다 비용 비중이 더 높음 (단가가 비싸서)')
print('→ 피크 시간대 전력을 줄이면 비용 절감 효과가 큽니다')

### 문제 4-2: 라인별·월별 전기요금 (5점)

In [ ]:
# 라인별·월별 전기요금 합계
monthly_cost = energy.groupby(['month', 'line'])['cost_won'].sum().unstack()

# 누적 바 차트
plt.figure(figsize=(12, 6))
monthly_cost.plot(kind='bar', stacked=True,
                  color=[line_colors['A라인'], line_colors['B라인'], line_colors['C라인']])

plt.xlabel('월')
plt.ylabel('전기요금 (원)')
plt.title('라인별·월별 전기요금', fontsize=13, fontweight='bold')
plt.xticks(range(6), ['1월', '2월', '3월', '4월', '5월', '6월'], rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 총 요금 및 라인별 비중
total_cost = energy.groupby('line')['cost_won'].sum()

print(f'=== 총 전기요금: {total_cost.sum():,.0f}원 ({total_cost.sum()/1e6:.1f}백만원) ===')
print()
for line in ['A라인', 'B라인', 'C라인']:
    pct = total_cost[line] / total_cost.sum() * 100
    print(f'{line}: {total_cost[line]:,.0f}원 ({pct:.1f}%)')

### 문제 4-3: 대기전력 비용 산출 (5점)

In [ ]:
# 비가동(대기) 시 전력과 비용
standby_data = energy[energy['is_operating'] == False]

standby_total_kwh = standby_data['power_kwh'].sum()
standby_total_cost = standby_data['cost_won'].sum()

print(f'=== 대기전력 현황 ===')
print(f'총 대기전력: {standby_total_kwh:,.0f} kWh')
print(f'총 대기전력 비용: {standby_total_cost:,.0f}원 ({standby_total_cost/1e6:.1f}백만원)')
print()

# 설비별 대기전력 비용 (상위 5개)
equip_standby = standby_data.groupby('equipment_id')['cost_won'].sum().sort_values(ascending=False)
print('=== 설비별 대기전력 비용 (상위 5) ===')
for eq, cost in equip_standby.head().items():
    print(f'  {eq}: {cost:,.0f}원')

In [ ]:
# A라인 절전모드 도입 시 절감 시뮬레이션
a_standby = standby_data[standby_data['equipment_id'].str.contains('EQ-A')]
a_standby_cost_6m = a_standby['cost_won'].sum()

# 절전모드 도입하면 50% 절감 가능하다고 가정
saving_50pct = a_standby_cost_6m * 0.5
saving_annual = saving_50pct * 2  # 연간

print(f'=== A라인 절전모드 도입 시뮬레이션 ===')
print(f'A라인 6개월 대기전력 비용: {a_standby_cost_6m:,.0f}원')
print(f'50% 절감 시 6개월 절감: {saving_50pct:,.0f}원')
print(f'연간 절감 예상: {saving_annual:,.0f}원 ({saving_annual/10000:,.0f}만원)')

### 문제 4-4: 피크 시프팅(Peak Shifting) 절감 시뮬레이션 (5점)

**피크 시프팅**: 비싼 시간대(최대부하)의 작업을 저렴한 시간대(경부하)로 옮겨서 요금 절감

In [ ]:
# 최대부하 시간대 데이터
peak_energy = energy[energy['time_zone'] == '최대부하']

# 평균 단가
peak_rate_avg = peak_energy['tariff_rate'].mean()  # 최대부하 평균 단가
offpeak_rate_avg = energy[energy['time_zone'] == '경부하']['tariff_rate'].mean()  # 경부하 평균 단가

total_peak_kwh = peak_energy['power_kwh'].sum()

print(f'최대부하 총 전력: {total_peak_kwh:,.0f} kWh')
print(f'최대부하 평균 단가: {peak_rate_avg:.0f}원/kWh')
print(f'경부하 평균 단가: {offpeak_rate_avg:.0f}원/kWh')
print(f'단가 차이: {peak_rate_avg - offpeak_rate_avg:.0f}원/kWh')

In [ ]:
# 시나리오별 절감 효과 계산
# 최대부하 전력의 10%, 20%, 30%를 경부하로 이전

print('=== 피크 시프팅 시나리오별 절감 효과 ===')
print()

rate_diff = peak_rate_avg - offpeak_rate_avg  # 단가 차이

for shift_pct in [10, 20, 30]:
    # 이전 전력량
    shifted_kwh = total_peak_kwh * shift_pct / 100
    
    # 절감액 = 이전 전력량 × 단가 차이
    cost_saving_6m = shifted_kwh * rate_diff
    cost_saving_annual = cost_saving_6m * 2
    
    print(f'{shift_pct}% 이전: '
          f'이전 전력 {shifted_kwh:,.0f}kWh → '
          f'연간 절감 {cost_saving_annual/10000:,.0f}만원')

print()
print('[코멘트]')
print('- 10% 이전: 야간 작업 일부 전환으로 현실적 실행 가능')
print('- 20% 이전: 교대 스케줄 조정 필요')
print('- 30% 이전: 대규모 변경이라 현실적 한계 있음')

---
## Part 5: 에너지 절감 대시보드 & 전략 제안 (15점)

### 문제 5-1: 에너지 분석 대시보드 (8점)

6개 차트를 한 화면에 모아서 대시보드를 만듭니다.

In [ ]:
# 2행 × 3열 = 6개 차트 대시보드
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# ===== (1,1) 라인별 월별 전력 바 차트 =====
ax = axes[0, 0]
monthly_line = energy.groupby(['month', 'line'])['power_kwh'].sum().unstack() / 1000  # MWh로 변환
x = np.arange(6)
w = 0.25  # 바 너비
for i, (line, color) in enumerate(line_colors.items()):
    ax.bar(x + i * w, monthly_line[line], w, color=color, label=line)
ax.set_xticks(x + w)
ax.set_xticklabels(['1월','2월','3월','4월','5월','6월'])
ax.set_ylabel('전력 (MWh)')
ax.set_title('라인별 월별 전력 사용량', fontweight='bold')
ax.legend(fontsize=8)

# ===== (1,2) 시간×요일 히트맵 =====
ax = axes[0, 1]
hm = energy.groupby(['weekday', 'hour'])['power_kwh'].mean().unstack()
sns.heatmap(hm, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'kWh'})
ax.set_yticklabels(['월','화','수','목','금','토','일'], rotation=0)
ax.set_xlabel('시간')
ax.set_title('시간×요일 전력 히트맵', fontweight='bold')

# ===== (1,3) 설비별 대기전력 =====
ax = axes[0, 2]
sb = standby_data.groupby('equipment_id')['power_kwh'].mean().sort_values(ascending=True)
inv_map = equip.set_index('equipment_id')['has_inverter'].to_dict()
colors_inv = ['#2ecc71' if inv_map.get(eq, False) else '#e74c3c' for eq in sb.index]
ax.barh(sb.index, sb.values, color=colors_inv)
ax.set_xlabel('평균 대기전력 (kWh)')
ax.set_title('설비별 대기전력', fontweight='bold')

# ===== (2,1) 원단위 월별 추이 =====
ax = axes[1, 0]
for line, color in line_colors.items():
    ax.plot(monthly_unit.index, monthly_unit[line], 'o-', color=color, linewidth=2, label=line)
ax.axvline(2.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(range(1,7))
ax.set_xticklabels(['1월','2월','3월','4월','5월','6월'])
ax.set_ylabel('원단위 (kWh/개)')
ax.set_title('월별 원단위 추이', fontweight='bold')
ax.legend(fontsize=8)

# ===== (2,2) 시간대별 요금 파이 차트 =====
ax = axes[1, 1]
tz_costs = energy.groupby('time_zone')['cost_won'].sum().reindex(tz_order)
tz_pct = tz_costs / tz_costs.sum() * 100
tz_colors_list = [tz_colors[tz] for tz in tz_order]
ax.pie(tz_pct, labels=tz_order, autopct='%1.1f%%', colors=tz_colors_list, startangle=90)
ax.set_title('시간대별 전기요금 비중', fontweight='bold')

# ===== (2,3) 외기온도 vs 전력 =====
ax = axes[1, 2]
s = energy.dropna().sample(3000, random_state=42)
for line, color in line_colors.items():
    mask = s['line'] == line
    ax.scatter(s[mask]['ambient_temp_c'], s[mask]['power_kwh'], alpha=0.3, s=8, c=color, label=line)
ax.set_xlabel('외기온도 (°C)')
ax.set_ylabel('전력 (kWh)')
ax.set_title('외기온도 vs 전력', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('에너지 사용 현황 대시보드 (2024년 상반기)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 문제 5-2: 에너지 절감 전략 제안 (7점)

In [ ]:
# 핵심 수치 정리
print('=' * 60)
print('에너지 절감 전략 수립을 위한 핵심 분석 결과')
print('=' * 60)

# [1] 라인별 현황
print('\n[1] 라인별 현황')
for line in ['A라인', 'B라인', 'C라인']:
    l_cost = energy[energy['line'] == line]['cost_won'].sum()
    l_unit = unit_clean[unit_clean['line'] == line]['energy_unit'].mean()
    print(f'  {line}: 비용 {l_cost/1e6:.1f}백만원, 원단위 {l_unit:.2f} kWh/개')

# [2] 대기전력
print(f'\n[2] 대기전력 비용: {standby_total_cost/1e6:.1f}백만원')

# [3] C라인 효과
print(f'\n[3] C라인 인버터 효과: 전력 {saving_rate:.1f}% 절감 (통계적 유의미)')

# [4] 절감 방안
print(f'\n[4] 절감 방안 요약')
print(f'  - A라인 절전모드: 연간 {saving_annual/10000:,.0f}만원 절감')
print(f'  - 피크 시프팅 10%: 연간 약 {total_peak_kwh * 0.1 * rate_diff * 2 / 10000:,.0f}만원 절감')

### 분석 결론

#### 1. 현재 에너지 사용 현황
- **A라인**: 전력의 약 43%, 원단위 가장 높음 (비효율적)
- **B라인**: 전력의 약 26%, 인버터 + 절전모드로 가장 효율적
- **C라인**: 전력의 약 31%, 3월 인버터 도입 후 개선 중

#### 2. 3대 낭비 포인트

| 낭비 유형 | 핵심 원인 |
|----------|----------|
| 대기전력 | A라인 인버터/절전모드 없음 |
| 피크 집중 | 최대부하 시간에 작업 집중 |
| 효율 격차 | A라인 노후 설비, 에너지등급 낮음 |

#### 3. C라인 인버터 도입 효과 (실증)
- 전력 약 20% 절감, 통계적으로 유의미 (p < 0.001)
- 대기전력도 크게 감소
- **A라인에도 도입 근거가 됨**

#### 4. 절감 방안 우선순위

| 순위 | 방안 | 효과 | 투자비 |
|-----|------|------|-------|
| 1 | A라인 절전모드 도입 | 높음 | 낮음 |
| 2 | 피크 시프팅 10~20% | 중간 | 낮음 |
| 3 | A라인 인버터 도입 | 높음 | 중간 |

---
## 수고하셨습니다!

### 학습 체크리스트
- [x] 시간별 에너지 데이터 전처리
- [x] 시간대별·요일별 전력 패턴 히트맵
- [x] 대기전력 분석
- [x] 에너지 원단위 계산 및 비교
- [x] C라인 인버터 효과 검증 (t-검정)
- [x] TOU 요금 기반 전기요금 산출
- [x] 피크 시프팅 시뮬레이션
- [x] 6패널 대시보드 구성
- [x] 에너지 절감 전략 제안